In [122]:
import os
from langchain.chat_models import init_chat_model


llm = init_chat_model(
    model="gemma4:31b-cloud",       # Name of the model pulled/hosted in Ollama
    model_provider="ollama",        # Specifies the Ollama provider 
    temperature=0.3,                # Controls randomness (lower means more focused or to the point)
    max_tokens=1000,                # Limits the maximum number of tokens in the response
)

# Define the system instruction to set assistant behavior
System = "you are a helpful Assistant"

# Initialize conversation history with the system instruction
messages = [
    {"role": "system", "content": System}
]

print("Chat started. Type 'quit' or 'exit' to end.\n")

# Start interactive continuous chat loop
while True:
    # Prompt the user for input
    user_input = input("You: ")
    
    if user_input.strip().lower() in ["quit", "exit"]:
        print("Goodbye!")
        break

    if not user_input.strip():
        continue

    messages.append({"role": "user", "content": user_input})

    print("AI: ", end="", flush=True)

    full_response = ""

    for chunk in llm.stream(messages):
        token = chunk.content
        full_response += token
        
        print(token, end="", flush=True)

    print("\n")

    messages.append({"role": "assistant", "content": full_response})

Chat started. Type 'quit' or 'exit' to end.

AI: Hello! How can I help you today?

AI: It's nice to meet you, Sultan! How is your day going so far? Is there anything I can help you with?

AI: AI is the ability of a computer or machine to mimic human intelligence to perform tasks and solve problems.

AI: Your name is Sultan.

Goodbye!


In [ ]:
import json
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gemma4:31b-cloud",
    model_provider="ollama",
    temperature=0.3,
    num_predict=1000,      # Ollama's name for max_tokens
    stop_=["```"],          # LangChain's name for stop sequences
)

messages = [
    {"role": "user", "content": "Give me three colours in JSON format with their hexadecimal codes."},
    {"role": "assistant", "content": "```json"},   # prefill: the start of the AI's reply
]

response = llm.invoke(messages)
response.content # repr shows exactly what came back

# data = json.loads(response.content)

''

In [58]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

llm = init_chat_model(
    model="gemini-2.5-flash",
    model_provider="google_genai",
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.3,
    stop_sequences=["```"]
)
messages=[{"role":"user","content":"generate 3 colursis json format with hexadecimal number"},{"role":"assistant","content":"```json"}]
response = llm.invoke(messages)
print(response.content)


[
  {
    "name": "Ruby Red",
    "hex": "#E0115F"
  },
  {
    "name": "Forest Green",
    "hex": "#228B22"
  },
  {
    "name": "Royal Blue",
    "hex": "#4169E1"
  }
]


## Prompt Evaluation

In [128]:
def generate_dataset():
  prompt="""
   Create an evaluation dataset for testing a prompt that extracts 4 fields from invoice text:
- invoice_number: string, or null if the text has no invoice number. Do not include the "#" symbol.
- date: string in DD-MM-YYYY format, or null if missing
- vendor: string (the company that sent the invoice), or null if missing
- total: number with no commas or currency symbol, or null if missing

Rules for the dataset:
1. Create exactly 10 items. Each invoice text is 1 to 3 lines.
2. Every invoice must have a different date and a different total.
3. Use a different date format across invoices, for example: "15/03/2023", "2024/06/15", "March 10, 2024", "29-FEB-2024", "3rd January 2027".
4. Ambiguous dates like 01-07-2025 are day first (1 July 2025).
5. Include each of these cases in exactly ONE invoice:
   - total written in words, e.g. "One Lakh Twenty Thousand Rupees"
   - total written in lakh, e.g. "Rs. 2.5 lakh"
   - total written with k, e.g. "2k USD"
   - total in Pakistani comma format, e.g. "Rs. 1,32,000"
   - no invoice number
   - no vendor name
   - an ambiguous date like 05-01-2024
6. Use a mix of currencies: Rs., PKR, $, USD, EUR.
7. When a field is missing, just leave it out of the text. Never write "no invoice number provided" or similar.
8. Do not reuse the example below.

Return only a JSON array, with no explanation and no code fences.

Example of one item:
[
  {
    "input": "Invoice INV-2041 from PakWheels, dated 5 Oct 2026. Total: PKR 85,000.",
    "expected": {
      "invoice_number": "INV-2041",
      "date": "05-10-2026",
      "vendor": "PakWheels",
      "total": 85000
    }
  }
]
    ```
    """
  return prompt
    



In [129]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gemma4:31b-cloud",
    model_provider="ollama",
    temperature=0.3,
    num_predict=3000,      # Ollama's name for max_tokens
)



response = llm.invoke(generate_dataset())
response.content



'[\n  {\n    "input": "Invoice #A-992 from TechCorp. 12/05/2023. Total: $450.50",\n    "expected": {\n      "invoice_number": "A-992",\n      "date": "12-05-2023",\n      "vendor": "TechCorp",\n      "total": 450.50\n    }\n  },\n  {\n    "input": "Bill 5501, 2024/11/20, Global Logistics. Total: Two Thousand Five Hundred Dollars",\n    "expected": {\n      "invoice_number": "5501",\n      "date": "20-11-2024",\n      "vendor": "Global Logistics",\n      "total": 2500\n    }\n  },\n  {\n    "input": "Invoice #B-102, March 15, 2022, Solar Systems. Total: Rs. 1.2 lakh",\n    "expected": {\n      "invoice_number": "B-102",\n      "date": "15-03-2022",\n      "vendor": "Solar Systems",\n      "total": 120000\n    }\n  },\n  {\n    "input": "INV-883, 29-FEB-2024, Cloud Services. Total: 5k USD",\n    "expected": {\n      "invoice_number": "INV-883",\n      "date": "29-02-2024",\n      "vendor": "Cloud Services",\n      "total": 5000\n    }\n  },\n  {\n    "input": "Invoice #C-441, 10-01-2025,

In [130]:
raw=response.content
cleaned_data=raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
dataset=json.loads(cleaned_data)
print(len(dataset))
with open("response.json", "w", encoding="utf-8") as file:
    json.dump(dataset, file, indent=2, ensure_ascii=False)

10


In [139]:
import os
import json
from pathlib import Path

# 1. Project folder mein jao
os.chdir(r"D:\Sultan\doc_Extractor")
print("Folder:", os.getcwd())

# 2. Folders banao
Path("eval").mkdir(exist_ok=True)
Path("prompts").mkdir(exist_ok=True)

# 3. Dataset file banao
dataset = [
  {"input": "Invoice #A-992 from TechCorp. 12/05/2023. Total: $450.50", "expected": {"invoice_number": "A-992", "date": "12-05-2023", "vendor": "TechCorp", "total": 450.5}},
  {"input": "Bill 5501, 2024/11/20, Global Logistics. Total: Two Thousand Five Hundred Dollars", "expected": {"invoice_number": "5501", "date": "20-11-2024", "vendor": "Global Logistics", "total": 2500}},
  {"input": "Invoice #B-102, March 15, 2022, Solar Systems. Total: Rs. 1.2 lakh", "expected": {"invoice_number": "B-102", "date": "15-03-2022", "vendor": "Solar Systems", "total": 120000}},
  {"input": "INV-883, 29-FEB-2024, Cloud Services. Total: 5k USD", "expected": {"invoice_number": "INV-883", "date": "29-02-2024", "vendor": "Cloud Services", "total": 5000}},
  {"input": "Invoice #C-441, 10-01-2025, FastTrack. Total: Rs. 1,45,000", "expected": {"invoice_number": "C-441", "date": "10-01-2025", "vendor": "FastTrack", "total": 145000}},
  {"input": "Dated 3rd January 2027, Apex Ltd. Total: EUR 890", "expected": {"invoice_number": None, "date": "03-01-2027", "vendor": "Apex Ltd", "total": 890}},
  {"input": "Invoice #D-772, 18-08-2021. Total: PKR 12300", "expected": {"invoice_number": "D-772", "date": "18-08-2021", "vendor": None, "total": 12300}},
  {"input": "Invoice #E-119, 05-01-2024, Zenith Co. Total: $77.25", "expected": {"invoice_number": "E-119", "date": "05-01-2024", "vendor": "Zenith Co", "total": 77.25}},
  {"input": "INV-009, 2023/07/14, Omega Inc. Total: EUR 1100.10", "expected": {"invoice_number": "INV-009", "date": "14-07-2023", "vendor": "Omega Inc", "total": 1100.1}},
  {"input": "Invoice #F-663, 22 Nov 2020, Prime Goods. Total: PKR 34000", "expected": {"invoice_number": "F-663", "date": "22-11-2020", "vendor": "Prime Goods", "total": 34000}},
]
with open("eval/dataset.json", "w", encoding="utf-8") as file:
    json.dump(dataset, file, indent=2, ensure_ascii=False)

# 4. Prompt file banao
prompt_v1 = """Extract the invoice number, date, vendor and total from this invoice.
Return JSON with these keys: invoice_number, date, vendor, total.

Invoice:
{invoice_text}
"""
with open("prompts/extract_v1.txt", "w", encoding="utf-8") as file:
    file.write(prompt_v1)

# 5. Check
print("dataset.json:", os.path.exists("eval/dataset.json"))
print("extract_v1.txt:", os.path.exists("prompts/extract_v1.txt"))

Folder: D:\Sultan\doc_Extractor
dataset.json: True
extract_v1.txt: True


In [ ]:
import json
from pathlib import Path

from langchain.chat_models import init_chat_model

PROMPT_FILE = "prompts/extract_v1.txt"
DATASET_FILE = "eval/dataset.json"
RESULTS_FILE = "eval/results_v1.json"
LIMIT = None       

with open(DATASET_FILE, encoding="utf-8") as file:
    dataset = json.load(file)

with open(PROMPT_FILE, encoding="utf-8") as file:
    prompt_template = file.read()

print("Invoices in dataset:", len(dataset))

llm = init_chat_model(
    model="gemma4:31b-cloud",
    model_provider="ollama",
    temperature=0,
    num_predict=1000,     
)


def clean_json(text: str) -> str:
    text = text.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```")
    return text.strip()


results = []
items = dataset if LIMIT is None else dataset[:LIMIT]

for number, item in enumerate(items, start=1):
    prompt = prompt_template.replace("{invoice_text}", item["input"])

    raw = llm.invoke(prompt).content

    try:
        actual = json.loads(clean_json(raw))
    except json.JSONDecodeError:
        actual = None

    results.append({
        "input": item["input"],
        "expected": item["expected"],
        "actual": actual,
        "raw": raw,           
    })

    status = "OK" if actual is not None else "JSON FAIL"
    print(f"[{number}/{len(items)}] {status}: {item['input'][:50]}...")

Path(RESULTS_FILE).parent.mkdir(exist_ok=True)
with open(RESULTS_FILE, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2, ensure_ascii=False)

print(f"\nSaved {len(results)} results to {RESULTS_FILE}")

Invoices in dataset: 10
[1/10] OK: Invoice #A-992 from TechCorp. 12/05/2023. Total: $...
[2/10] OK: Bill 5501, 2024/11/20, Global Logistics. Total: Tw...
[3/10] OK: Invoice #B-102, March 15, 2022, Solar Systems. Tot...
[4/10] OK: INV-883, 29-FEB-2024, Cloud Services. Total: 5k US...
[5/10] OK: Invoice #C-441, 10-01-2025, FastTrack. Total: Rs. ...
[6/10] OK: Dated 3rd January 2027, Apex Ltd. Total: EUR 890...
[7/10] OK: Invoice #D-772, 18-08-2021. Total: PKR 12300...
[8/10] OK: Invoice #E-119, 05-01-2024, Zenith Co. Total: $77....
[9/10] OK: INV-009, 2023/07/14, Omega Inc. Total: EUR 1100.10...
[10/10] OK: Invoice #F-663, 22 Nov 2020, Prime Goods. Total: P...

Saved 10 results to eval/results_v1.json


In [143]:
import json

RESULTS_FILE = "eval/results_v1.json"
FIELDS = ["invoice_number", "date", "vendor", "total"]

with open(RESULTS_FILE, encoding="utf-8") as file:
    results = json.load(file)


def clean(value):
    """Space hatao, chhote huroof. None ko None hi rehne do."""
    if value is None:
        return None
    return str(value).strip().lower()


def is_correct(field, expected, actual):
    if expected is None and actual is None:
        return True
    if expected is None or actual is None:
        return False

    if field == "vendor":
        return clean(expected).rstrip(".") == clean(actual).rstrip(".")
    if field == "invoice_number":
        return clean(expected) == clean(actual)

    return expected == actual


correct = {field: 0 for field in FIELDS}
failures = []

for number, result in enumerate(results, start=1):
    expected = result["expected"]
    actual = result["actual"] or {field: None for field in FIELDS}

    for field in FIELDS:
        exp_value = expected.get(field)
        act_value = actual.get(field)   

        if is_correct(field, exp_value, act_value):
            correct[field] += 1
        else:
            failures.append((number, field, exp_value, act_value))

total_checks = len(results) * len(FIELDS)
total_correct = sum(correct.values())

print("=" * 60)
print(f"SCORE: {total_correct}/{total_checks} ({total_correct / total_checks:.0%})")
print("=" * 60)
for field in FIELDS:
    print(f"  {field:<15} {correct[field]}/{len(results)}")

print(f"\nFAILURES ({len(failures)}):")
for number, field, exp_value, act_value in failures:
    print(f"  Invoice {number:<2} | {field:<14} | expected: {str(exp_value):<12} | got: {act_value}")

SCORE: 23/40 (57%)
  invoice_number  10/10
  date            3/10
  vendor          10/10
  total           0/10

FAILURES (17):
  Invoice 1  | date           | expected: 12-05-2023   | got: 12/05/2023
  Invoice 1  | total          | expected: 450.5        | got: $450.50
  Invoice 2  | date           | expected: 20-11-2024   | got: 2024/11/20
  Invoice 2  | total          | expected: 2500         | got: Two Thousand Five Hundred Dollars
  Invoice 3  | date           | expected: 15-03-2022   | got: March 15, 2022
  Invoice 3  | total          | expected: 120000       | got: Rs. 1.2 lakh
  Invoice 4  | date           | expected: 29-02-2024   | got: 29-FEB-2024
  Invoice 4  | total          | expected: 5000         | got: 5k USD
  Invoice 5  | total          | expected: 145000       | got: Rs. 1,45,000
  Invoice 6  | date           | expected: 03-01-2027   | got: 3rd January 2027
  Invoice 6  | total          | expected: 890          | got: EUR 890
  Invoice 7  | total          | expected

In [148]:
import json
from pathlib import Path

from langchain.chat_models import init_chat_model

PROMPT_FILE = "prompts/extract_v2.txt"
DATASET_FILE = "eval/dataset.json"
RESULTS_FILE = "eval/results_v2.json"
LIMIT = None       

with open(DATASET_FILE, encoding="utf-8") as file:
    dataset = json.load(file)

with open(PROMPT_FILE, encoding="utf-8") as file:
    prompt_template = file.read()

print("Invoices in dataset:", len(dataset))

llm = init_chat_model(
    model="gemma4:31b-cloud",
    model_provider="ollama",
    temperature=0,
    num_predict=1000,     
)


def clean_json(text: str) -> str:
    text = text.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```")
    return text.strip()


results = []
items = dataset if LIMIT is None else dataset[:LIMIT]

for number, item in enumerate(items, start=1):
    prompt = prompt_template.replace("{invoice_text}", item["input"])

    raw = llm.invoke(prompt).content

    try:
        actual = json.loads(clean_json(raw))
    except json.JSONDecodeError:
        actual = None

    results.append({
        "input": item["input"],
        "expected": item["expected"],
        "actual": actual,
        "raw": raw,           
    })

    status = "OK" if actual is not None else "JSON FAIL"
    print(f"[{number}/{len(items)}] {status}: {item['input'][:50]}...")

Path(RESULTS_FILE).parent.mkdir(exist_ok=True)
with open(RESULTS_FILE, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2, ensure_ascii=False)

print(f"\nSaved {len(results)} results to {RESULTS_FILE}")

Invoices in dataset: 10
[1/10] OK: Invoice #A-992 from TechCorp. 12/05/2023. Total: $...
[2/10] OK: Bill 5501, 2024/11/20, Global Logistics. Total: Tw...
[3/10] OK: Invoice #B-102, March 15, 2022, Solar Systems. Tot...
[4/10] OK: INV-883, 29-FEB-2024, Cloud Services. Total: 5k US...
[5/10] OK: Invoice #C-441, 10-01-2025, FastTrack. Total: Rs. ...
[6/10] OK: Dated 3rd January 2027, Apex Ltd. Total: EUR 890...
[7/10] OK: Invoice #D-772, 18-08-2021. Total: PKR 12300...
[8/10] OK: Invoice #E-119, 05-01-2024, Zenith Co. Total: $77....
[9/10] OK: INV-009, 2023/07/14, Omega Inc. Total: EUR 1100.10...
[10/10] OK: Invoice #F-663, 22 Nov 2020, Prime Goods. Total: P...

Saved 10 results to eval/results_v2.json


In [149]:
import json

RESULTS_FILE = "eval/results_v2.json"
FIELDS = ["invoice_number", "date", "vendor", "total"]

with open(RESULTS_FILE, encoding="utf-8") as file:
    results = json.load(file)


def clean(value):
    """Space hatao, chhote huroof. None ko None hi rehne do."""
    if value is None:
        return None
    return str(value).strip().lower()


def is_correct(field, expected, actual):
    if expected is None and actual is None:
        return True
    if expected is None or actual is None:
        return False

    if field == "vendor":
        return clean(expected).rstrip(".") == clean(actual).rstrip(".")
    if field == "invoice_number":
        return clean(expected) == clean(actual)

    return expected == actual


correct = {field: 0 for field in FIELDS}
failures = []

for number, result in enumerate(results, start=1):
    expected = result["expected"]
    actual = result["actual"] or {field: None for field in FIELDS}

    for field in FIELDS:
        exp_value = expected.get(field)
        act_value = actual.get(field)   

        if is_correct(field, exp_value, act_value):
            correct[field] += 1
        else:
            failures.append((number, field, exp_value, act_value))

total_checks = len(results) * len(FIELDS)
total_correct = sum(correct.values())

print("=" * 60)
print(f"SCORE: {total_correct}/{total_checks} ({total_correct / total_checks:.0%})")
print("=" * 60)
for field in FIELDS:
    print(f"  {field:<15} {correct[field]}/{len(results)}")

print(f"\nFAILURES ({len(failures)}):")
for number, field, exp_value, act_value in failures:
    print(f"  Invoice {number:<2} | {field:<14} | expected: {str(exp_value):<12} | got: {act_value}")

SCORE: 40/40 (100%)
  invoice_number  10/10
  date            10/10
  vendor          10/10
  total           10/10

FAILURES (0):
